In [1]:
import pandas as pd
import numpy as np
from functools import reduce
from pathlib import Path

# Load data
data_path = Path('..') / 'data' / 'cleaned' / 'amazon_products_sales_data_cleaned.csv'
df = pd.read_csv(data_path)
df['data_collected_at'] = pd.to_datetime(df['data_collected_at'])

print(f"Loaded {len(df)} products across {df['product_category'].nunique()} categories")

Loaded 42675 products across 15 categories


In [2]:
# Stream-like filtering and mapping to find high-value premium products
# Using lambda expressions and method chaining (Python's stream equivalent)

premium_products = (
    df[['product_title', 'product_rating', 'discounted_price', 'total_reviews', 'product_category']]
    .query('product_rating >= 4.5 and discounted_price >= 100 and total_reviews >= 1000')  # Filter
    .assign(value_score=lambda x: x['product_rating'] * np.log1p(x['total_reviews']))  # Map with lambda
    .sort_values('value_score', ascending=False)  # Sort
    .head(10)  # Limit
)

print("\nTop 10 Premium Products by Value Score:")
print(premium_products[['product_title', 'product_rating', 'discounted_price', 'value_score']].to_string())


Top 10 Premium Products by Value Score:
                                                                                                                                                                                            product_title  product_rating  discounted_price  value_score
4796                                                                                                                                                  Nintendo Switch with Neon Blue and Neon Red Joy‑Con             4.8            299.00    54.235981
2912  Crucial 32GB DDR4 RAM Kit (2x16GB), 3200MHz (PC4-25600) CL22 Laptop Memory, SODIMM 260-Pin, Downclockable to 2933/2666MHz, Compatible with 13th Gen Intel Core and AMD Ryzen 7000 - CT2K16G4SFRA32A             4.8            107.99    53.540736
3719                                                                                                      SanDisk SSD PLUS 2TB Internal SSD - SATA III 6 Gb/s, 2.5"/7mm, Up to 545 MB/s - SDSSDA-2T00-G26           

In [3]:
# Multi-stage functional aggregation pipeline
# Demonstrate groupBy, map, reduce operations

category_metrics = (
    df.groupby('product_category')
    .agg({
        'discounted_price': ['mean', 'median', 'std'],
        'product_rating': 'mean',
        'total_reviews': 'sum',
        'product_title': 'count'
    })
    .round(2)
)

# Flatten column names and apply transformations
category_metrics.columns = ['_'.join(col).strip('_') for col in category_metrics.columns]
category_metrics = category_metrics.rename(columns={'product_title_count': 'product_count'})

# Use lambda to calculate price efficiency score
category_metrics['price_efficiency'] = category_metrics.apply(
    lambda row: row['product_rating_mean'] / (row['discounted_price_mean'] / 100),
    axis=1
).round(3)

print("\nCategory Performance Metrics:")
print(category_metrics.sort_values('price_efficiency', ascending=False))


Category Performance Metrics:
                     discounted_price_mean  discounted_price_median  \
product_category                                                      
Power & Batteries                    53.51                    22.25   
Chargers & Cables                   125.69                    34.99   
Storage                             136.36                    93.99   
Headphones                          136.77                   129.99   
Other Electronics                   145.08                    59.24   
Printers & Scanners                 158.18                    80.05   
Phones                              159.04                    63.32   
Gaming                              163.94                   154.23   
TV & Display                        177.21                    85.99   
Speakers                            178.15                   134.18   
Networking                          217.75                   129.99   
Wearables                           259.33    

In [4]:
# Functional reduce operation: Calculate cumulative discount impact across categories
# Using reduce to combine discount statistics

discount_stats = df.groupby('product_category')['discount_percentage'].apply(list).to_dict()

# Define reducer function for combining discount metrics
def discount_reducer(acc, item):
    category, discounts = item
    valid_discounts = [d for d in discounts if pd.notna(d) and d > 0]
    if valid_discounts:
        acc[category] = {
            'avg_discount': np.mean(valid_discounts),
            'max_discount': np.max(valid_discounts),
            'products_with_discount': len(valid_discounts)
        }
    return acc

# Apply reduce operation
discount_analysis = reduce(discount_reducer, discount_stats.items(), {})
discount_df = pd.DataFrame(discount_analysis).T.round(2)

print("\nDiscount Analysis by Category (using reduce):")
print(discount_df.sort_values('avg_discount', ascending=False))


Discount Analysis by Category (using reduce):
                     avg_discount  max_discount  products_with_discount
Headphones                  31.48         80.59                   437.0
Networking                  30.73         63.88                   384.0
Chargers & Cables           30.00         68.77                   771.0
Wearables                   28.47         47.95                    47.0
Speakers                    26.92         82.50                   353.0
Phones                      25.55         85.42                  1982.0
Laptops                     23.34         77.33                  2400.0
Storage                     21.07         50.05                   899.0
Other Electronics           18.65         61.96                  1434.0
Gaming                      18.49         69.45                   133.0
TV & Display                16.73         66.67                   521.0
Printers & Scanners         16.21         65.12                   497.0
Power & Batteries

In [5]:
# Advanced lambda and filter operations: Best seller analysis
# Chaining multiple functional operations

# Filter function for best sellers
is_best_seller = lambda x: x == 'Best Seller'
has_high_rating = lambda x: x >= 4.5

# Apply filters and transformations
best_seller_insights = (
    df[df['is_best_seller'].apply(is_best_seller)]
    .assign(
        rating_category=lambda x: x['product_rating'].apply(
            lambda r: 'Excellent' if r >= 4.7 else 'Very Good' if r >= 4.5 else 'Good'
        )
    )
    .groupby(['product_category', 'rating_category'])
    .agg({
        'product_title': 'count',
        'discounted_price': 'mean',
        'total_reviews': 'median'
    })
    .rename(columns={'product_title': 'count'})
    .round(2)
)

print("\nBest Seller Distribution by Rating Category:")
print(best_seller_insights)


Best Seller Distribution by Rating Category:
                                     count  discounted_price  total_reviews
product_category    rating_category                                        
Cameras             Excellent            6            154.17         3241.0
                    Good                 3             96.63         1899.0
                    Very Good            1             22.99         3224.0
Chargers & Cables   Excellent            8             45.48        11738.5
                    Good                10             58.53        18835.5
                    Very Good            6             71.00        17586.0
Gaming              Excellent            3            155.05        34440.0
                    Good                 3            264.22         2186.0
                    Very Good            2            189.99        80131.5
Headphones          Good                 1             82.00        16761.0
Laptops             Excellent            9